# Large Reasoning Model Evaluation for Silkome Reasoning

By: Jay Siri (jaysiri@mit.edu)

This notebook evaluates finetuned LLMs that reason about silk fiber mechanical properties.

**Interpretation:**
- Visualize attention to important parts of input

**Evaluation:**
- Tokenize test set and do inference
- Extract <strength, toughness> from responses, clip and round
- Compute error metrics (RMSE, MAE, Pearson Correlation) and visualize heatmaps


### **Set Up and Load Model**


In [ ]:
!pip install torchao --upgrade

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
import json
import pandas as pd

from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from datasets import load_dataset
from peft import PeftModel
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.feature_selection import r_regression

In [ ]:
EXAMPLE_INPUT = """Your task is to consider a protein sequence in FASTA format, along with the family, genus, species, and protein category to predict the strength and toughness of silk fibers produced by the spider.

Please analyze the following data:

Protein sequence: SSSAYSGTSTGGSSVSQSQPIISSAPVYFNAQTLTSSLASSLQSDRALNFISSGQLSASDVSTSVSSAVAQTLGISQSSVQNIISQQMSSVRTGASSSSVSQAIANAVSSAVQASGAATPGQEQSIAQRVYSSISTYLSQLISQRTAPAPAPAPRPAPMPAPAPRPAPMPAPAPRPRPAPAPRPAPVYAPAPVVSQIQAAASSQSSAQQSSFAQAQQSAYAQSQQSSSAYSGASTGGSSVSQSQPIISSAPVYFNAQTLTSSLASSLQSDRALNFISSGQLSASDVSTSVSSAVAQTLGISQSSVQNIISQQMSSVRTGASSSSVSQAIANAVSSAVQASGAATPGQEQSIAQRVYSSISTYLSQLISQRTAPAPAPAPRPAPMPAPAPRPAPMPAPAPRPRPAPAPRPASVYAPAPVVSQIQAAASSQSSSQQSSFAQAQQSAYAQSQQSSSAYSGASTGGSSVSQSQPIISSAPVYFNAQTLTSSLASSLQSDRALNFISSGQLSASDVSTSVSSAVAQTLGISQSSVQNIISQQMSSVRTGASSSSVSQAIANAVSSAVQASGAATPGQEQSIAQRVYSSISTYLSQLISQRTAPAPAPAPRPAPMPAPAPRPAPMPAPAPRPRPAPAPRPAPVYAPAPVVSQIQAAASSQSSAQQSSFAQAQQSAYAQSQQSSSAYSGASTGGSSVSQSQPIISSAPVYFNAQTLTSSLASSLQSDRALNFISSGQLSASDVSTSVSSAVAQTLGISQSSVQNIISQQMSSVRTGASSSSVSQAIANAVSSAVQASGAATPGQEQSIAQRVYSSISTYLSQLISQRTAPAPAPAPRPAPMPAPAPRPAPMPAPAPRPRPAPAPRPAPVYAPAPVVSQIQAAASSQSSSQQSSFAQAQQSAYAQSQQSSSAYSGASTGGSSVSQSQPIASSAPVYFNTQTLSSSLASSLQSDRALNFISSGQLSAADVSTGVSSAVAQALGISQSSVQNIISQQMSSIRTGASSSSVSQAIANAVSSAVQASGAAAPGQEQGIAQSVYSAISTYLSQLISQRTAPAPAPAPRPLPAPMPAPAPRPRPAPAPRPAPIYAPAPVVSQLQTASSSQSSAQQNSFAQSQQSAFAQSQQSSIAAQSQQRQSANAYSTAPSFAGSQVQQAAAAQSQVSASSFSSGSSPGIGSSPSSSAAFSAPISSAPYAPSVVASSSSSVGPSSGSALAASAAQQLTSAAANQRIAALSNSLRSAVAGGQVNYGALSNSLASAASQIQRSSGLSKSEVLVEALLETLAALLDSLSTSGSSSSQFAQAVLQAFA
Family: Araneidae
Genus: Eriophora
Species: pustulosa
Protein category: PySp.

Now predict the strength and toughness of the silk fibers, each in a range between 0.0 and 1.0."""

SYSTEM_PROMPT_SILK_1 = """Explain and reason step-by-step how you got those values, and do not report a default value.

Then report the strength and toughness in this exact XML format:
<answer>
{"strength": <float in the range [0, 1]>, "toughness": <float in the range [0, 1]>}
</answer>
"""

In [ ]:
# Choose a Hugging Face pretrained model name
ADAPTER_NAME ='jayyysiri/silkome_reasoning_v1.0' # Public model adapter name from HF
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)
model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(model, ADAPTER_NAME)

In [ ]:
def model_pred(input_str):
  """Helper to generate a prediction given an input prompt."""
  generated = torch.tensor(tokenizer.encode(input_str)).unsqueeze(0).to(model.device)
  sample_outputs = model.generate(
                    inputs=generated,
                    max_new_tokens=2500,
                    min_new_tokens=50,
                    pad_token_id=tokenizer.pad_token_id,
                    temperature=0.5,
                    use_cache=False
                    )
  decoded_text = tokenizer.decode(sample_outputs[-1], skip_special_tokens=True)
  return decoded_text

### **Visualize Attention on Example Input**

Use the model to do inference on the example input. Then visualize the most attention-varied heads of the last layer.

In [ ]:
# Tokenize the input text
input_text = EXAMPLE_INPUT + "\n" + SYSTEM_PROMPT_SILK_1
inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=True)

# Get the input IDs and decode them to see the tokens, including special tokens
input_ids = inputs['input_ids'][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

In [ ]:
# Configure model to output attention weights
model.config.output_attentions = True
outputs = model(**inputs.to(model.device), output_attentions=True, return_dict=True)
attention_weights = outputs.attentions

print(f"Number of attention layers: {len(attention_weights)}")
print(f"Shape of attention weights for the first layer (batch_size, num_heads, sequence_length, sequence_length): {attention_weights[0].shape}")

In [ ]:
# From the last layer of the model, get the heads with the highest scores
scores = []
layer_index = model.config.num_hidden_layers - 1

for layer in range(len(attention_weights)):
    for head in range(attention_weights[layer_index].shape[1]):
        attn = attention_weights[layer_index][0, head]
        scores.append((layer_index, head, attn.var().item()))

scores.sort(key=lambda x: x[2], reverse=True)
ordered_head_indexes = []

for i, (layer, head, score) in enumerate(scores):
  if head not in ordered_head_indexes:
    ordered_head_indexes.append(head)

print(f"Last-layer heads with highest attention scores (largest to smallest): {ordered_head_indexes}")

In [ ]:
def plot_heatmap_for_layer_head(layer_index, head_index, ax=None):
  """
  Visualize the attention weights of the top 20 tokens in a specific layer and head.

  Args:
    layer_index: The index of the layer to visualize.
    head_index: The index of the head to visualize.
  """
  N = 20 # number of tokens

  # Extract attention scores for the selected layer and head
  # attention_weights is a tuple of tensors, each tensor is (batch_size, num_heads, sequence_length, sequence_length)
  attn = attention_weights[layer_index][0, head_index].detach().cpu()   # [seq_len, seq_len]
  seq_len = attn.shape[0]
  vals, idx = torch.topk(attn.flatten(), N)
  rows = idx // seq_len
  cols = idx % seq_len

  heatmap = torch.zeros_like(attn)
  heatmap[rows, cols] = vals
  important_tokens = torch.unique(torch.cat([rows, cols]))
  sub = heatmap[important_tokens][:, important_tokens]

  tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
  labels = [tokens[i] for i in important_tokens]

  # Replace non-English special delimiters for readability
  labels_cleaned = []
  for label in labels:
    temp = label.replace("Ġ", " ")
    temp = temp.replace("Ċ", " ")
    labels_cleaned.append(temp)

  # Use provided axis or current axis
  if ax is None:
      ax = plt.gca()

  ax.imshow(sub.float().numpy())

  ax.set_xticks(range(len(labels)))
  ax.set_xticklabels(labels_cleaned, rotation=90)

  ax.set_yticks(range(len(labels)))
  ax.set_yticklabels(labels_cleaned)
  ax.set_title(f"Top {N} Attention Connections for Layer {layer_index}, Head {head_index}")
  return im

In [ ]:
# Plot top N heads using helper
N_TO_PLOT = 6

fig, axes = plt.subplots(2, N_TO_PLOT // 2, figsize=(30, 10))
for head_idx, ax_obj in zip(ordered_head_indexes[:N_TO_PLOT], axes.ravel()):
  im = plot_heatmap_for_layer_head(model.config.num_hidden_layers - 1, head_idx, ax=ax_obj)

fig.colorbar(im, ax=axes.ravel().tolist(), label="Attention weight")
plt.tight_layout()
plt.show()

### **Inference**

Load in the Silkome test split and use the model to do inference on the test split prompts. Extract responses as floats and handle clipping or edge cases in responses.

In [ ]:
# Read in Silkome data from GitHub
SILKOME_TRAIN_URL = "https://github.com/jaypsiri/Silkome-Reasoning/raw/refs/heads/main/data/silkome_reasoning_dataset/train/data-00000-of-00001.arrow"
SILKOME_TEST_URL= "https://github.com/jaypsiri/Silkome-Reasoning/raw/refs/heads/main/data/silkome_reasoning_dataset/test/data-00000-of-00001.arrow"
silk = load_dataset("arrow", data_files={'train': SILKOME_TRAIN_URL, 'test': SILKOME_TEST_URL})
silk_test = silk.get("test")

# Generate responses for Silkome test
model_responses_silk_test = []
print(f"Performing inference on {len(silk_test.data["input"])} inputs ...")
for i in range(len(silk_test.data["input"]))[:1]:
  decoded_text = model_pred(silk_test.data["input"].to_pylist()[i] + "\n" + SYSTEM_PROMPT_SILK_1)
  expected = silk_test.data["output"].to_pylist()[i]
  model_responses_silk_test.append([decoded_text, expected])

In [ ]:
# Extract numeric strength and toughness out of the responses
prompt_len = len(SYSTEM_PROMPT_SILK_1)

def find_nth(text_to_search: str, text_to_find: str, occurance_n: int) -> int:
    start = text_to_search.find(text_to_find)
    while start >= 0 and occurance_n > 1:
        start = text_to_search.find(text_to_find, start+len(text_to_find))
        occurance_n -= 1
    return start

def try_to_get_answer(input_str: str):
  """
  Helper to try to extract answer enclosed in <answer> tag
  """
  try:
    start_index = find_nth(input_str, "<answer>", 2)
    end_index = find_nth(input_str, "</answer>", 2)
    if start_index == -1 or end_index == -1:
      return None
    extracted_ans = input_str[start_index + len("<answer>"): end_index]
    return extracted_ans
  except:
    return None

pred_strength = []
pred_toughness = []
true_strength = []
true_toughness = []

for i in range(len(model_responses_silk_test)):
  attempted_ans = try_to_get_answer(model_responses_silk_test[i][0])
  try:
    attempted_ans = attempted_ans.strip()
    attempted_ans = json.loads(attempted_ans)
    pred_strength.append(float(attempted_ans.get("strength", None)))
    pred_toughness.append(float(attempted_ans.get("toughness", None)))
  except:
    pred_strength.append(None)
    pred_toughness.append(None)

  expected_output = json.loads(model_responses_silk_test[i][1])
  true_strength.append(float(expected_output.get("strength", None)))
  true_toughness.append(float(expected_output.get("toughness", None)))

### **Metrics and Visualization of Quantitative Results**

Visualize the true vs. predicted strength and toughness. Calculate error metrics.

In [ ]:
def clip_numeric_answers(predictions):
  for i in range(len(predictions)):
    if predictions[i]:
      predictions[i] = round(predictions[i], 1)
      if predictions[i] > 1:
        predictions[i] = 1
      if predictions[i] < 0:
        predictions[i] = 0
  return predictions

clip_numeric_answers(pred_strength)
clip_numeric_answers(pred_toughness)

In [ ]:
def compute_metrics_and_plot(raw_pred, raw_true, name):
  pred = []
  true = []
  for i in range(len(raw_pred)):
    if raw_pred[i] and raw_true[i]:
      pred.append(raw_pred[i])
      true.append(raw_true[i])

  print(f"Metrics for {name} \n ==================")
  print(f"R-Squared: {r2_score(true, pred)}")
  print(f"RMSE: {root_mean_squared_error(true, pred)}")
  print(f"MAE: {mean_absolute_error(true, pred)}")

  # Plot heatmap of extracted responses
  counts, xedges, yedges, im = plt.hist2d(
      pred,
      true,
      bins=11,
      range=[[0, 1], [0, 1]],
      cmap='viridis'
  )

  plt.colorbar(im, label='Count')
  plt.xlabel(f'Predicted {name}')
  plt.ylabel(f'True {name}')
  plt.title(f'Predicted vs. True {name} (n={len(pred)})')
  plt.show()

In [ ]:
compute_metrics_and_plot(pred_strength, true_strength, "Strength")

In [ ]:
compute_metrics_and_plot(pred_toughness, true_toughness, "Toughness")